In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("mobility_report_countries.csv")

In [3]:
df.dtypes ## change the date column to datetime and make it the index, maybe aggregate by week? make sure it follows the stringency index

country                   object
region                    object
date                      object
retail and recreation    float64
grocery and pharmacy     float64
parks                    float64
transit stations         float64
workplaces               float64
residential              float64
dtype: object

In [4]:

#df = df[(df["country"] == "United States") | (df["country"] == "Singapore") | (df["country"] == "South Korea")]
df = df[df["region"] == "Total"]
df.head()

,country,region,date,retail and recreation,grocery and pharmacy,parks,transit stations,workplaces,residential
0,United Arab Emirates,Total,2020-02-15,0.0,4.0,5.0,0.0,2.0,1.0
1,United Arab Emirates,Total,2020-02-16,1.0,4.0,4.0,1.0,2.0,1.0
2,United Arab Emirates,Total,2020-02-17,-1.0,1.0,5.0,1.0,2.0,1.0
3,United Arab Emirates,Total,2020-02-18,-2.0,1.0,5.0,0.0,2.0,1.0
4,United Arab Emirates,Total,2020-02-19,-2.0,0.0,4.0,-1.0,2.0,1.0


In [5]:
df.head() ## date from 2/15/2020 to 8/19/2022

,country,region,date,retail and recreation,grocery and pharmacy,parks,transit stations,workplaces,residential
0,United Arab Emirates,Total,2020-02-15,0.0,4.0,5.0,0.0,2.0,1.0
1,United Arab Emirates,Total,2020-02-16,1.0,4.0,4.0,1.0,2.0,1.0
2,United Arab Emirates,Total,2020-02-17,-1.0,1.0,5.0,1.0,2.0,1.0
3,United Arab Emirates,Total,2020-02-18,-2.0,1.0,5.0,0.0,2.0,1.0
4,United Arab Emirates,Total,2020-02-19,-2.0,0.0,4.0,-1.0,2.0,1.0


In [6]:
## want changes in mobility, we care when mobility as a whole increases or decreases
import datetime

df.reset_index()

,index,country,region,date,retail and recreation,grocery and pharmacy,parks,transit stations,workplaces,residential
0,0,United Arab Emirates,Total,2020-02-15,0.0,4.0,5.0,0.0,2.0,1.0
1,1,United Arab Emirates,Total,2020-02-16,1.0,4.0,4.0,1.0,2.0,1.0
2,2,United Arab Emirates,Total,2020-02-17,-1.0,1.0,5.0,1.0,2.0,1.0
3,3,United Arab Emirates,Total,2020-02-18,-2.0,1.0,5.0,0.0,2.0,1.0
4,4,United Arab Emirates,Total,2020-02-19,-2.0,0.0,4.0,-1.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...
123218,1728607,Zimbabwe,Total,2022-08-15,122.0,126.0,273.0,160.0,82.0,19.0
123219,1728608,Zimbabwe,Total,2022-08-16,123.0,134.0,252.0,147.0,80.0,19.0
123220,1728609,Zimbabwe,Total,2022-08-17,129.0,141.0,267.0,167.0,81.0,19.0
123221,1728610,Zimbabwe,Total,2022-08-18,132.0,136.0,261.0,159.0,76.0,19.0


In [7]:
df["date"] = pd.to_datetime(df["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 123223 entries, 0 to 1728611
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   country                123223 non-null  object        
 1   region                 123223 non-null  object        
 2   date                   123223 non-null  datetime64[ns]
 3   retail and recreation  120950 non-null  float64       
 4   grocery and pharmacy   120878 non-null  float64       
 5   parks                  120725 non-null  float64       
 6   transit stations       121835 non-null  float64       
 7   workplaces             122308 non-null  float64       
 8   residential            120549 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 9.4+ MB


In [8]:
#df_time = df.set_index(pd.DatetimeIndex(df["date"])).drop("date", axis = 1)
#df_time.head()

In [9]:
out_cols = [
    'retail and recreation',
    'grocery and pharmacy',
    'parks',
    'transit stations',
    'workplaces'
]

df['total_out'] = df[out_cols].mean(axis=1) 

In [10]:
df['week'] = df['date'].dt.to_period('W')

In [11]:
weekly_df = df.groupby(['week',"country"]).agg({
    'total_out': 'mean',
    'residential': 'mean'
}).reset_index()

In [12]:
weekly_df

,week,country,total_out,residential
0,2020-02-10/2020-02-16,Afghanistan,-2.00,2.5
1,2020-02-10/2020-02-16,Angola,-0.60,1.5
2,2020-02-10/2020-02-16,Antigua and Barbuda,-1.70,3.0
3,2020-02-10/2020-02-16,Argentina,-3.40,1.0
4,2020-02-10/2020-02-16,Aruba,3.50,-1.0
...,...,...,...,...
17766,2022-08-15/2022-08-21,Venezuela,46.68,15.8
17767,2022-08-15/2022-08-21,Vietnam,11.70,8.8
17768,2022-08-15/2022-08-21,Yemen,141.84,14.2
17769,2022-08-15/2022-08-21,Zambia,118.88,27.6


In [13]:
s = pd.read_csv("OxCGRT_compact_national_v1.csv")

In [14]:
s.head()

,CountryName,CountryCode,RegionName,RegionCode,Jurisdiction,Date,C1M_School closing,C1M_Flag,C2M_Workplace closing,C2M_Flag,...,V3_Vaccine Financial Support (summary),V4_Mandatory Vaccination (summary),ConfirmedCases,ConfirmedDeaths,MajorityVaccinated,PopulationVaccinated,StringencyIndex_Average,GovernmentResponseIndex_Average,ContainmentHealthIndex_Average,EconomicSupportIndex
0,Aruba,ABW,NaN,NaN,NAT_TOTAL,20200101,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
1,Aruba,ABW,NaN,NaN,NAT_TOTAL,20200102,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
2,Aruba,ABW,NaN,NaN,NAT_TOTAL,20200103,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
3,Aruba,ABW,NaN,NaN,NAT_TOTAL,20200104,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
4,Aruba,ABW,NaN,NaN,NAT_TOTAL,20200105,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0


In [15]:
s['Date'] = pd.to_datetime(s['Date'], format='%Y%m%d')

In [16]:
#strin = s[(s["CountryName"] == "United States") | (s["CountryName"] == "Singapore")| (s["CountryName"] == "South Korea")].copy()

In [17]:
strin = s.copy()
strin['week'] = strin['Date'].dt.to_period('W')
strin.head()

,CountryName,CountryCode,RegionName,RegionCode,Jurisdiction,Date,C1M_School closing,C1M_Flag,C2M_Workplace closing,C2M_Flag,...,V4_Mandatory Vaccination (summary),ConfirmedCases,ConfirmedDeaths,MajorityVaccinated,PopulationVaccinated,StringencyIndex_Average,GovernmentResponseIndex_Average,ContainmentHealthIndex_Average,EconomicSupportIndex,week
0,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-01,0.0,NaN,0.0,NaN,...,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0,2019-12-30/2020-01-05
1,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-02,0.0,NaN,0.0,NaN,...,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0,2019-12-30/2020-01-05
2,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-03,0.0,NaN,0.0,NaN,...,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0,2019-12-30/2020-01-05
3,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-04,0.0,NaN,0.0,NaN,...,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0,2019-12-30/2020-01-05
4,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-05,0.0,NaN,0.0,NaN,...,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0,2019-12-30/2020-01-05


In [18]:
weekly_stringency = strin.groupby(['week', "CountryName"])['StringencyIndex_Average'].mean().reset_index()

In [19]:
weekly_stringency.head(20)

,week,CountryName,StringencyIndex_Average
0,2019-12-30/2020-01-05,Afghanistan,0.0
1,2019-12-30/2020-01-05,Albania,0.0
2,2019-12-30/2020-01-05,Algeria,0.0
3,2019-12-30/2020-01-05,Andorra,0.0
4,2019-12-30/2020-01-05,Angola,0.0
5,2019-12-30/2020-01-05,Argentina,0.0
6,2019-12-30/2020-01-05,Aruba,0.0
7,2019-12-30/2020-01-05,Australia,0.0
8,2019-12-30/2020-01-05,Austria,0.0
9,2019-12-30/2020-01-05,Azerbaijan,0.0


In [21]:
import altair as alt
countries = [
    "United States",
    "China",
    "New Zealand",
    "Sweden",
    "Brazil",
    "Germany"
]

# Make copies so you do not overwrite your originals
mob = weekly_df.copy()
stri = weekly_stringency.copy()

# Convert week to datetime if still Period
if str(mob["week"].dtype).startswith("period"):
    mob["week"] = mob["week"].dt.start_time

if str(stri["week"].dtype).startswith("period"):
    stri["week"] = stri["week"].dt.start_time

# Rename country column in stringency df so both match
stri = stri.rename(columns={"CountryName": "country"})

# Filter to selected countries
mob = mob[mob["country"].isin(countries)]
stri = stri[stri["country"].isin(countries)]

# Keep only the columns we need
mob = mob[["week", "country", "residential"]]
stri = stri[["week", "country", "StringencyIndex_Average"]]

# Merge
final_df = pd.merge(mob, stri, on=["week", "country"], how="inner")

# Reshape to long format for two-line plotting
plot_df = final_df.melt(
    id_vars=["week", "country"],
    value_vars=["residential", "StringencyIndex_Average"],
    var_name="measure",
    value_name="value"
)

# Cleaner labels
plot_df["measure"] = plot_df["measure"].replace({
    "residential": "Residential Mobility",
    "StringencyIndex_Average": "Stringency Index"
})

# Faceted line chart with shared axes
chart = alt.Chart(plot_df).mark_line().encode(
    x=alt.X("week:T", title="Week"),
    y=alt.Y(
        "value:Q",
        title="Value",
        scale=alt.Scale(domain=[plot_df["value"].min(), plot_df["value"].max()])
    ),
    color=alt.Color("measure:N", title="Measure"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("measure:N", title="Measure"),
        alt.Tooltip("value:Q", title="Value", format=".2f")
    ]
).properties(
    width=250,
    height=180
).facet(
    column=alt.Column("country:N", title="Country")
).properties(
    title="Weekly Stringency Index and Residential Mobility by Country"
)

chart

alt.FacetChart(...)

In [22]:
import pandas as pd
import altair as alt

countries = [
    "United States",
    "China",
    "New Zealand",
    "Sweden",
    "Brazil",
    "Germany"
]

plot_df = weekly_df[weekly_df["country"].isin(countries)].copy()

# Make sure week is datetime
if str(plot_df["week"].dtype).startswith("period"):
    plot_df["week"] = plot_df["week"].dt.start_time
else:
    plot_df["week"] = pd.to_datetime(plot_df["week"])

# Smoothed chart source
smooth_chart = alt.Chart(plot_df).transform_window(
    smoothed_residential="mean(residential)",
    frame=[-2, 2],
    groupby=["country"]
)

lines = smooth_chart.mark_line(size=2).encode(
    x=alt.X("week:T", title="Week"),
    y=alt.Y("smoothed_residential:Q", title="Residential Mobility (% Change from Baseline)"),
    color=alt.Color("country:N", title="Country"),
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

points = smooth_chart.mark_circle(size=40, opacity=0).encode(
    x="week:T",
    y="smoothed_residential:Q",
    color="country:N",
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

chart = (lines + points).properties(
    title="Smoothed Residential Mobility Over Time by Country",
    width=800,
    height=450
).interactive()

chart

alt.LayerChart(...)

In [23]:
import pandas as pd
import altair as alt

# final_df should already contain:
# week, country, residential, StringencyIndex_Average

# Base scatter
points = alt.Chart(final_df).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X("StringencyIndex_Average:Q", title="Stringency Index"),
    y=alt.Y("residential:Q", title="Residential Mobility (% Change from Baseline)"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("StringencyIndex_Average:Q", title="Stringency", format=".2f"),
        alt.Tooltip("residential:Q", title="Residential", format=".2f")
    ]
)

# Regression line
trend = alt.Chart(final_df).transform_regression(
    "StringencyIndex_Average",
    "residential"
).mark_line(size=3, color="black").encode(
    x="StringencyIndex_Average:Q",
    y="residential:Q"
)

# R^2 labels: regression transform with params=True returns rSquared
r2_text = alt.Chart(final_df).transform_regression(
    "StringencyIndex_Average",
    "residential",
    params=True
).transform_calculate(
    label="'R² = ' + format(datum.rSquared, '.2f')"
).mark_text(
    align="left",
    baseline="top",
    dx=5,
    dy=5,
    fontSize=12
).encode(
    text="label:N",
    x=alt.value(5),
    y=alt.value(5)
)

chart = (points + trend + r2_text).properties(
    width=220,
    height=220
).facet(
    column=alt.Column("country:N", title="Country")
).properties(
    title="Residential Mobility vs Stringency by Country"
)

chart

alt.FacetChart(...)

## Visualization #6:

#### Linear Regression:

In [24]:
s.head() ## gov support, stringency, response, containment, economic support

,CountryName,CountryCode,RegionName,RegionCode,Jurisdiction,Date,C1M_School closing,C1M_Flag,C2M_Workplace closing,C2M_Flag,...,V3_Vaccine Financial Support (summary),V4_Mandatory Vaccination (summary),ConfirmedCases,ConfirmedDeaths,MajorityVaccinated,PopulationVaccinated,StringencyIndex_Average,GovernmentResponseIndex_Average,ContainmentHealthIndex_Average,EconomicSupportIndex
0,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-01,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
1,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-02,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
2,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-03,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
3,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-04,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0
4,Aruba,ABW,NaN,NaN,NAT_TOTAL,2020-01-05,0.0,NaN,0.0,NaN,...,0,NaN,0.0,0.0,NV,0.0,0.0,0.0,0.0,0.0


In [91]:
owid = pd.read_csv("owid-covid-data.csv")
owid.columns
### total_cases_per_million
### total_deaths_per_million

Index(['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       't

##### plan:
y = new_deaths_per_million

X = [["stringency_index", "residential mobility"]]

other factors: median_age, aged_65_older

health systems: hospital_beds_per_thousand

economic: gdp_per_capita

population: population_density

In [92]:
## Location (change to country)
## date (change to datetime)
## stringency (stringency_index)
## new_deaths_per_million

### then convert to weekly

owid["date"] = pd.to_datetime(owid["date"])
owid_man = owid.copy()

cols = ["location", "date", "stringency_index", 'new_deaths_per_million']
owid_man = owid_man.loc[:, cols]
owid_man

,location,date,stringency_index,new_deaths_per_million
0,Afghanistan,2020-01-05,0.0,0.0
1,Afghanistan,2020-01-06,0.0,0.0
2,Afghanistan,2020-01-07,0.0,0.0
3,Afghanistan,2020-01-08,0.0,0.0
4,Afghanistan,2020-01-09,0.0,0.0
...,...,...,...,...
429430,Zimbabwe,2024-07-31,NaN,0.0
429431,Zimbabwe,2024-08-01,NaN,0.0
429432,Zimbabwe,2024-08-02,NaN,0.0
429433,Zimbabwe,2024-08-03,NaN,0.0


In [35]:
# % missing per column per country
missing_by_country = owid.groupby('location').apply(lambda x: x.isna().mean())

missing_by_country.head(20)

C:\Users\Emma\AppData\Local\Temp\ipykernel_67900\2709186876.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_country = owid.groupby('location').apply(lambda x: x.isna().mean())


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
location,,,,,,,,,,,,,,,,,,,,,
Afghanistan,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,0.0,0.0,0.0,0.0,0.0,1.000000,1.000000,1.000000,1.000000
Africa,0.0,1.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,1.0,1.0,1.0,1.0,0.0,1.000000,1.000000,1.000000,1.000000
Albania,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,0.0,1.0,0.0,0.0,0.0,0.0,0.971326,0.971326,0.971326,0.971326
Algeria,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,0.0,0.0,0.0,0.0,0.0,0.0,0.971326,0.971326,0.971326,0.971326
American Samoa,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,1.0,1.0,0.0,1.0,0.0,1.000000,1.000000,1.000000,1.000000
Andorra,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,0.0,1.0,1.0,0.0,0.0,0.0,0.985663,0.985663,0.985663,0.985663
Angola,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,0.0,1.0,0.0,0.0,0.0,1.000000,1.000000,1.000000,1.000000
Anguilla,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,1.0,1.0,0.0,1.0,0.0,1.000000,1.000000,1.000000,1.000000
Antigua and Barbuda,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002987,0.000000,0.000000,0.002987,...,1.0,1.0,0.0,0.0,0.0,0.0,0.985663,0.985663,0.985663,0.985663


In [38]:
key_cols = [
    'new_deaths_smoothed_per_million',
    'stringency_index',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand'
]

missing_by_country = owid.groupby('location')[key_cols].apply(lambda x: x.isna().mean())
missing_by_country.head(20)

,new_deaths_smoothed_per_million,stringency_index,gdp_per_capita,median_age,hospital_beds_per_thousand
location,,,,,
Afghanistan,0.002987,0.347670,0.0,0.0,0.0
Africa,0.002987,1.000000,1.0,1.0,1.0
Albania,0.002987,0.347670,0.0,0.0,0.0
Algeria,0.002987,0.347670,0.0,0.0,0.0
American Samoa,0.002987,1.000000,1.0,1.0,1.0
Andorra,0.002987,0.347670,1.0,1.0,1.0
Angola,0.002987,0.347670,0.0,0.0,1.0
Anguilla,0.002987,1.000000,1.0,1.0,1.0
Antigua and Barbuda,0.002987,1.000000,0.0,0.0,0.0


In [93]:
selected_countries = [
    "United States", "China", "New Zealand", "Sweden", "Brazil", "Germany",
    "United Kingdom", "France", "Italy", "Spain",
    "Canada", "Australia", "Japan", "South Korea", "Singapore",
    "Netherlands", "Belgium", "Denmark", "Norway"
]
owid_filter = owid[owid['location'].isin(selected_countries)].copy()

In [94]:
print("Number of countries:", owid_filter['location'].nunique())
print(owid_filter['location'].unique())

Number of countries: 19
['Australia' 'Belgium' 'Brazil' 'Canada' 'China' 'Denmark' 'France'
 'Germany' 'Italy' 'Japan' 'Netherlands' 'New Zealand' 'Norway'
 'Singapore' 'South Korea' 'Spain' 'Sweden' 'United Kingdom'
 'United States']


In [95]:
owid_filter = owid_filter[~owid_filter['iso_code'].str.startswith('OWID')]

In [97]:
cols_needed = [
    'location', 'date',
    'new_deaths_smoothed_per_million',
    'stringency_index',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density'
]

owid_filter = owid_filter[cols_needed]
owid_filter.info()
owid_filter.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 31816 entries, 21776 to 405124
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   location                         31816 non-null  object        
 1   date                             31816 non-null  datetime64[ns]
 2   new_deaths_smoothed_per_million  30524 non-null  float64       
 3   stringency_index                 20748 non-null  float64       
 4   gdp_per_capita                   31816 non-null  float64       
 5   median_age                       31816 non-null  float64       
 6   hospital_beds_per_thousand       31816 non-null  float64       
 7   population_density               31816 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 2.2+ MB


location                               0
date                                   0
new_deaths_smoothed_per_million     1292
stringency_index                   11068
gdp_per_capita                         0
median_age                             0
hospital_beds_per_thousand             0
population_density                     0
dtype: int64

In [98]:
owid_filter = owid_filter.dropna(subset=['new_deaths_smoothed_per_million'])
owid_filter.shape

(30524, 8)

In [99]:
owid_filter = owid_filter.sort_values(['location', 'date'])

owid_filter['stringency_index'] = owid_filter.groupby('location')['stringency_index'].ffill()

In [100]:
owid_filter.isna().sum()
owid_filter = owid_filter.dropna()
owid_filter.shape

(30524, 8)

#### Lag/Timing Variables

In [101]:
owid_filter = owid_filter.sort_values(["location", "date"])
owid_filter['stringency_lag_2'] = owid_filter.groupby('location')['stringency_index'].shift(2)
owid_filter['stringency_lag_3'] = owid_filter.groupby('location')['stringency_index'].shift(3)

In [102]:
owid_filter = owid_filter.dropna(subset=['stringency_lag_2'])

In [107]:
owid_filter = owid_filter.copy()
owid_filter['first_case_date'] = owid_filter.groupby('location')['date'].transform(
    lambda x: x[owid_filter.loc[x.index, 'new_deaths_smoothed_per_million'] > 0].min()
)
owid_filter['days_since_outbreak'] = (
    owid_filter['date'] - owid_filter['first_case_date']
).dt.days

early_stringency = (
    owid_filter[owid_filter['days_since_outbreak'] <= 30]
    .groupby('location')['stringency_index']
    .mean()
    .reset_index()
    .rename(columns={'stringency_index': 'early_stringency'})
)

owid_filter
owid_filter = owid_filter.merge(early_stringency, on='location', how='left')

In [108]:
df_model = owid_filter[[
    'location',
    'date',
    'new_deaths_smoothed_per_million',
    'stringency_lag_2',
    'early_stringency',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density'
]].dropna()

In [109]:
import statsmodels.api as sm

X = df_model['stringency_lag_2']
y = df_model['new_deaths_smoothed_per_million']

X = sm.add_constant(X)

model1 = sm.OLS(y, X).fit()
print(model1.summary())

                                   OLS Regression Results                                  
Dep. Variable:     new_deaths_smoothed_per_million   R-squared:                       0.128
Model:                                         OLS   Adj. R-squared:                  0.128
Method:                              Least Squares   F-statistic:                     4477.
Date:                             Mon, 06 Apr 2026   Prob (F-statistic):               0.00
Time:                                     20:53:10   Log-Likelihood:                -64311.
No. Observations:                            30486   AIC:                         1.286e+05
Df Residuals:                                30484   BIC:                         1.286e+05
Df Model:                                        1                                         
Covariance Type:                         nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.

In [110]:
X = df_model[
    [
        'stringency_lag_2',
        'early_stringency',
        'gdp_per_capita',
        'median_age',
        'hospital_beds_per_thousand',
        'population_density'
    ]
]

y = df_model['new_deaths_smoothed_per_million']

X = sm.add_constant(X)

model2 = sm.OLS(y, X).fit()
print(model2.summary())

                                   OLS Regression Results                                  
Dep. Variable:     new_deaths_smoothed_per_million   R-squared:                       0.195
Model:                                         OLS   Adj. R-squared:                  0.195
Method:                              Least Squares   F-statistic:                     1229.
Date:                             Mon, 06 Apr 2026   Prob (F-statistic):               0.00
Time:                                     20:53:29   Log-Likelihood:                -63096.
No. Observations:                            30486   AIC:                         1.262e+05
Df Residuals:                                30479   BIC:                         1.263e+05
Df Model:                                        6                                         
Covariance Type:                         nonrobust                                         
                                 coef    std err          t      P>|t|      [0.0

In [111]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

X = df_model[
    [
        'stringency_lag_2',
        'early_stringency',
        'gdp_per_capita',
        'median_age',
        'hospital_beds_per_thousand',
        'population_density'
    ]
]

y = df_model['new_deaths_smoothed_per_million']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tree = DecisionTreeRegressor(max_depth=4)
tree.fit(X_train, y_train)

print("Train R²:", tree.score(X_train, y_train))
print("Test R²:", tree.score(X_test, y_test))

Train R²: 0.4282910693832247
Test R²: 0.4600144382358876
